## API exploration

Exploring the API of "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations" to assess feasibility of a ML project.

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from hubeau.ades import GestionnairePiezometrie

In [4]:
gestionnaire = GestionnairePiezometrie(dossier_sortie="data")
# Liste de vos forages (remplacez par les vrais codes de votre bassin versant)
forages_bassin = [
        "BSS001EUKK", # forage à Saffré
        "BSS001PYMM", # Mâcon (issu de la doc)
        "BSS001PYKW"  # Un autre exemple
    ]

df = gestionnaire.telecharger_bassin_versant(forages_bassin,
                                            pause_secondes=3 )

--- Début du traitement pour 3 forages ---
Téléchargement de BSS001EUKK...
  -> Succès : 2009 mesures sauvegardées dans piezo_BSS001EUKK.csv
  [Pause de 3s pour ne pas surcharger le serveur...]
Téléchargement de BSS001PYMM...
  -> Succès : 6549 mesures sauvegardées dans piezo_BSS001PYMM.csv
  [Pause de 3s pour ne pas surcharger le serveur...]
Téléchargement de BSS001PYKW...
  -> Succès : 1463 mesures sauvegardées dans piezo_BSS001PYKW.csv
--- Traitement du bassin versant terminé ! ---


### Data Engineering

**To do list**

- Creer un dataframe à partir du JSON. Refactoriser le code de la fonction en module
- Explorer les données vendéennes :
examples : CHallans BSS001LADS   Entité  121
        niort
        la roche s yon BSS001MHUZ  Entité
        Cholet   BSS001JWAE          Entité       181
          ruffec BSS001RRGC

- Essayer de charger la vendée en entier.
- Moyenne des piezos de vendée.
- Sortir les coordonnées géographiques Lambert 93

### Emplacement géographiques des différentes stations de mesure.

La liste des stations existentes est stockée différemment des données piezometrique. Une autre database JSON

On peut rechercher la base par département, par entité BDlisa.

In [6]:
from hubeau.entite import Entite

vendee = Entite()

print("--- Recherche par département ---")
print(s85 := vendee.rechercher_stations(code_dep = "85"))


--- Recherche par département ---
Recherche des stations avec les paramètres : {'format': 'json', 'size': 5000, 'code_departement': '85'}...
-> 56 stations trouvées !
['BSS001JNKL', 'BSS001MELY', 'BSS001PGFU', 'BSS001NNMN', 'BSS001NLTS', 'BSS001NKVQ', 'BSS001MDPN', 'BSS001NMRJ', 'BSS001MKDU', 'BSS001LADS', 'BSS001PGNV', 'BSS001KYWM', 'BSS001PEED', 'BSS001KYWN', 'BSS001KYQK', 'BSS001KYUA', 'BSS001MDTW', 'BSS001PEPF', 'BSS001MHUF', 'BSS001KYXH', 'BSS001PFCH', 'BSS001PFFH', 'BSS001KYXF', 'BSS001PEFC', 'BSS001KYXG', 'BSS001PEXS', 'BSS001PFNL', 'BSS001LCHH', 'BSS001KYWZ', 'BSS001NMFQ', 'BSS001PFLX', 'BSS001KYXA', 'BSS001NKUA', 'BSS001JNKN', 'BSS001MKMT', 'BSS001NLEZ', 'BSS001MHUZ', 'BSS001MFCH', 'BSS001JPGZ', 'BSS001LDCR', 'BSS001NMFS', 'BSS001PFNU', 'BSS001NHNA', 'BSS001LDCD', 'BSS001NMZR', 'BSS001NLSE', 'BSS001JPGP', 'BSS001NLJH', 'BSS001NMSL', 'BSS001NMQC', 'BSS004AXJB', 'BSS004AXHK', 'BSS001PGPK', 'BSS001NKUB', 'BSS001KYQR', 'BSS001NKBJ']


In [7]:
vendee.donnees_brutes

[{'code_bss': '05068X0028/SP010',
  'urn_bss': 'http://services.ades.eaufrance.fr/pointeau/05068X0028/SP010',
  'date_debut_mesure': '1990-12-11',
  'date_fin_mesure': '2026-05-18',
  'code_commune_insee': '85083',
  'nom_commune': "L'Épine",
  'x': -2.240128861,
  'y': 46.990355837,
  'codes_bdlisa': ['121AF01'],
  'urns_bdlisa': ['http://reseau.eaufrance.fr/geotraitements/bdlisa/files/entite/121AF01.pdf'],
  'geometry': {'type': 'Point',
   'crs': {'type': 'name',
    'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
   'coordinates': [-2.24012886096318, 46.9903558366809]},
  'bss_id': 'BSS001JNKL',
  'altitude_station': '2.25',
  'nb_mesures_piezo': 12827,
  'code_departement': '85',
  'nom_departement': 'Vendée',
  'libelle_pe': "L'Epine forage du terrain-neuf (L'Epine-85)",
  'profondeur_investigation': 16.5,
  'codes_masse_eau_edl': ['GG036'],
  'noms_masse_eau_edl': ['Ile de Noirmoutier'],
  'urns_masse_eau_edl': ['http://www.sandre.eaufrance.fr/geo/MasseDEauSouterraine/

In [8]:
vendee.generer_df().head()

-> Catalogue structuré en DataFrame (56 lignes, 22 colonnes).


/home/charourou/projects/hydrosense/hubeau/entite.py:90: FutureWarning: Parsing 'CEST' as tzlocal (dependent on system timezone) is deprecated and will raise in a future version. Pass the 'tz' keyword or call tz_localize after construction instead
  return pd.to_datetime(str(valeur), utc=True)
/home/charourou/projects/hydrosense/hubeau/entite.py:90: FutureWarning: Parsing 'CET' as tzlocal (dependent on system timezone) is deprecated and will raise in a future version. Pass the 'tz' keyword or call tz_localize after construction instead
  return pd.to_datetime(str(valeur), utc=True)


,code_bss,urn_bss,date_debut_mesure,date_fin_mesure,code_commune_insee,nom_commune,x,y,codes_bdlisa,urns_bdlisa,...,altitude_station,nb_mesures_piezo,code_departement,nom_departement,libelle_pe,profondeur_investigation,codes_masse_eau_edl,noms_masse_eau_edl,urns_masse_eau_edl,date_maj
0,05068X0028/SP010,http://services.ades.eaufrance.fr/pointeau/050...,1990-12-11 00:00:00+00:00,2026-05-18 00:00:00+00:00,85083,L'Épine,-2.240129,46.990356,121AF01,http://reseau.eaufrance.fr/geotraitements/bdli...,...,2.25,12827,85,Vendée,L'Epine forage du terrain-neuf (L'Epine-85),16.5,GG036,Ile de Noirmoutier,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,NaT
1,05604X0162/SF1,http://services.ades.eaufrance.fr/pointeau/056...,2011-01-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,85189,Notre-Dame-de-Riez,-1.884899,46.768020,121AF01,http://reseau.eaufrance.fr/geotraitements/bdli...,...,7.0,5397,85,Vendée,Forage du Breuil( Notre Dame de Riez - 85),24.0,GG033,Sables et calcaires du bassin tertiaire de Jau...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,NaT
2,06101X0202/SP1,http://services.ades.eaufrance.fr/pointeau/061...,1987-07-10 00:00:00+00:00,2026-05-18 00:00:00+00:00,85168,Oulmes,-0.687081,46.408744,"360AA07,362AF01",http://reseau.eaufrance.fr/geotraitements/bdli...,...,10.2,12201,85,Vendée,"forage du Gd Nati (Oulmes,85)",41.0,GG042,Calcaires et marnes du Lias et Dogger du Sud-V...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,NaT
3,05867X0152/SR1,http://services.ades.eaufrance.fr/pointeau/058...,1987-08-31 00:00:00+00:00,2026-02-24 00:00:00+00:00,85092,Fontenay-le-Comte,-0.836079,46.452581,None,,...,4.71,11883,85,Vendée,"forage du Gros Noyer (Fontenay le Comte,85)",25.0,GG042,Calcaires et marnes du Lias et Dogger du Sud-V...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,NaT
4,05861X0113/SF2,http://services.ades.eaufrance.fr/pointeau/058...,1987-09-20 00:00:00+00:00,2026-05-18 00:00:00+00:00,85290,Thiré,-1.013552,46.541679,None,,...,51.93,11784,85,Vendée,"forage de la la Ville Morte(Thiré,85)",53.0,GG042,Calcaires et marnes du Lias et Dogger du Sud-V...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,NaT


In [ ]:
print("\n--- Recherche par entité hydrogéologique ---")
stations_meme_nappe = Entite().rechercher_stations(code_bdlisa="113AC10")
print(f"\nVoici les 5 premières stations de la nappe : {stations_meme_nappe[:5]}")



--- Recherche par entité hydrogéologique ---
Recherche des stations avec les paramètres : {'format': 'json', 'size': 5000, 'codes_bdlisa': '113AC10'}...
-> 5000 stations trouvées !

Voici les 5 premières stations de la nappe : ['BSS000RPJR', 'BSS000NXYM', 'BSS001ZJPH', 'BSS000LVRN', 'BSS002PJXA']


list

In [25]:
print("\n--- Recherche par département 44 ---")
s44 = Entite().rechercher_stations(code_dep="44")


collecteur = GestionnairePiezometrie(dossier_sortie= '/home/charourou/projects/hydrosense/data/')
derniere_collecte = []



--- Recherche par département 44 ---
Recherche des stations avec les paramètres : {'format': 'json', 'size': 5000, 'code_departement': '44'}...
-> 58 stations trouvées !


In [28]:

for nom_piezo in s44[:]:
    df = collecteur.telecharger_forage(nom_piezo)
    if not type(df) == bool:
        derniere_collecte.append(df['date_mesure'].iloc[-1])
    else:
        s44.remove(nom_piezo)

Info: données BSS001EUMW déja présente
Info: données BSS001GPCB déja présente
Info: données BSS001EUZL déja présente
Info: données BSS001JPMN déja présente
Info: données BSS001EUJD déja présente
Info: données BSS001ETCD déja présente
Info: données BSS001DJDQ déja présente
Téléchargement de BSS001DJDC...
  -> Erreur lors du traitement de BSS001DJDC : No columns to parse from file
Info: données BSS001ETCE déja présente
Info: données BSS001DKCX déja présente
Info: données BSS001JRHS déja présente
Info: données BSS001HBQB déja présente
Info: données BSS001EUZK déja présente
Info: données BSS001JPMP déja présente
Info: données BSS001HBQA déja présente
Info: données BSS001DMWC déja présente
Info: données BSS001JPMQ déja présente
Info: données BSS001EUHC déja présente
Info: données BSS001DJDP déja présente
Info: données BSS001EUKK déja présente
Info: données BSS001ESVX déja présente
Info: données BSS001ESVY déja présente
Info: données BSS001EUZH déja présente
Info: données BSS001DKCW déja pré

In [29]:
derniere_collecte

['2026-05-28',
 '2026-05-28',
 '2026-05-28',
 '2026-05-28',
 '2010-05-28',
 '2024-03-22',
 '2015-12-14',
 '2026-05-27',
 '2005-06-18',
 '2026-05-28',
 '1998-03-16',
 '2026-05-27',
 '2026-05-28',
 '2026-05-27',
 '2026-05-27',
 '2026-05-28',
 '2026-04-30',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2020-06-25',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-04-30',
 '2026-05-27',
 '2026-04-30',
 '2026-05-27',
 '2026-05-27',
 '2005-10-18',
 '2026-05-27',
 '2025-12-01',
 '2008-03-06',
 '2026-05-28',
 '2026-05-27',
 '2019-02-11',
 '2026-04-06',
 '2026-05-26',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-26',
 '2026-05-27',
 '2026-05-22',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-27',
 '2026-05-28',
 '2026-05-28',
 '2026-05-27',
 '2026-05-27',
 '2026-05-26']

### Plots et un peu d'exploration de données. 


- plot dans le temps

- comparaison des données entre deux piezometres

- valeurs moyennes sur plusieurs entitées

- geopandas pour une representation graphique